# GPU experiments (Colab)

**Before running:** `Runtime` → `Change runtime type` → Hardware accelerator: **T4 GPU** (or A100 / L4 if you have Colab Pro). Then `Runtime` → `Run all`.

Each section writes its numbers into a `results` dict, and the last cell saves `results.json` so you can download it or copy it to Drive.


## 1. What GPU did we get?

In [ ]:
import subprocess, torch, time, json, platform
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

assert torch.cuda.is_available(), "No GPU. Change runtime type to a GPU and re-run."
dev = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
results = {
    "gpu": props.name,
    "vram_gb": round(props.total_memory / 1e9, 2),
    "cuda": torch.version.cuda,
    "torch": torch.__version__,
    "python": platform.python_version(),
}
results

## 2. Matmul throughput (TFLOPS) in fp32 / fp16 / bf16

In [ ]:
def bench_matmul(n=8192, dtype=torch.float32, iters=20):
    a = torch.randn(n, n, device=dev, dtype=dtype)
    b = torch.randn(n, n, device=dev, dtype=dtype)
    for _ in range(3):              # warmup
        a @ b
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        a @ b
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / iters
    return 2 * n**3 / dt / 1e12    # TFLOPS

results["matmul_tflops"] = {}
for name, dt in [("fp32", torch.float32), ("fp16", torch.float16), ("bf16", torch.bfloat16)]:
    try:
        tf = bench_matmul(dtype=dt)
        results["matmul_tflops"][name] = round(tf, 1)
        print(f"{name}: {tf:6.1f} TFLOPS")
    except Exception as e:
        results["matmul_tflops"][name] = None
        print(f"{name}: unsupported ({type(e).__name__})")
torch.cuda.empty_cache()

## 3. Memory bandwidth (GB/s)

In [ ]:
def bench_bandwidth(n_bytes=2 * 1024**3, iters=20):
    x = torch.empty(n_bytes // 4, device=dev, dtype=torch.float32)
    y = torch.empty_like(x)
    for _ in range(3):
        y.copy_(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        y.copy_(x)
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / iters
    return 2 * n_bytes / dt / 1e9    # read + write

bw = bench_bandwidth()
results["mem_bandwidth_gbs"] = round(bw, 0)
print(f"{bw:.0f} GB/s")
torch.cuda.empty_cache()

## 4. End-to-end training: small CNN on MNIST, 1 epoch, CPU vs GPU

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train = datasets.MNIST("/content/data", train=True, download=True, transform=tfm)
test  = datasets.MNIST("/content/data", train=False, download=True, transform=tfm)
train_dl = DataLoader(train, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test,  batch_size=1000, num_workers=2, pin_memory=True)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 32, 3); self.c2 = nn.Conv2d(32, 64, 3)
        self.f1 = nn.Linear(9216, 128); self.f2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.relu(self.c1(x)); x = F.max_pool2d(F.relu(self.c2(x)), 2)
        x = torch.flatten(x, 1); x = F.relu(self.f1(x))
        return self.f2(x)

def train_one_epoch(device, use_amp=False):
    torch.manual_seed(0)
    model = Net().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler(enabled=use_amp)
    model.train(); t0 = time.perf_counter()
    for xb, yb in train_dl:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            loss = F.cross_entropy(model(xb), yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    if device.type == "cuda": torch.cuda.synchronize()
    train_s = time.perf_counter() - t0
    model.eval(); correct = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    return train_s, correct / len(test)

results["mnist_1epoch"] = {}
for label, device, amp in [("gpu_fp32", dev, False), ("gpu_amp_fp16", dev, True)]:
    s, acc = train_one_epoch(device, amp)
    results["mnist_1epoch"][label] = {"train_seconds": round(s, 1), "test_acc": round(acc, 4)}
    print(f"{label:13s}  {s:6.1f}s  acc={acc:.4f}")

# CPU run on a subset so it finishes quickly; scale up if you want the full comparison.
import itertools
cpu = torch.device("cpu"); torch.manual_seed(0); m = Net(); o = torch.optim.Adam(m.parameters())
t0 = time.perf_counter(); n_batches = 20
for xb, yb in itertools.islice(train_dl, n_batches):
    o.zero_grad(); F.cross_entropy(m(xb), yb).backward(); o.step()
cpu_est = (time.perf_counter() - t0) / n_batches * len(train_dl)
results["mnist_1epoch"]["cpu_fp32_estimated"] = {"train_seconds": round(cpu_est, 1)}
print(f"cpu_fp32 (est) {cpu_est:6.1f}s  (extrapolated from {n_batches} batches)")

## 5. Your experiment

Drop your own code below. `dev`, `results`, and PyTorch are already set up. Add whatever you measure to `results[...]` so it ends up in the saved JSON.

In [ ]:
# TODO: your experiment here
# results["my_experiment"] = {...}

## 6. Save results

In [ ]:
print(json.dumps(results, indent=2))
with open("/content/results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved to /content/results.json  (Files panel on the left → right-click → Download)")

# Optional: copy to Google Drive so it survives the runtime being recycled.
# from google.colab import drive; drive.mount("/content/drive")
# import shutil; shutil.copy("/content/results.json", "/content/drive/MyDrive/results.json")